# Use case: multi-site variability in the UCI Heart Disease data with `dashi`

**Scenario.** The UCI Heart Disease data were collected at four hospitals: Cleveland (USA), Budapest (Hungary),
Long Beach VA (USA) and Zurich/Basel (Switzerland). Most published models use the Cleveland subset only. Suppose
we want to reuse such a model at another hospital, or pool the four sites to train a larger one. Before doing
so we need to know *how comparable the sites are*.

This notebook answers that question with the multi-source tools of `dashi`:

| Question | `dashi` tool |
|---|---|
| Does the disease prevalence change by site? (**prior shift**) | Data Source Maps (DSMs) + MSV metrics on the label |
| Do the patient characteristics change by site? (**covariate shift**) | Univariate and multivariate DSMs + MSV metrics |
| Do the differences remain within each diagnosis? (**conditional / concept shift**) | Label-conditional DSMs |
| Does a model trained at one site transfer to the others? | Multi-batch models and performance matrices |

Two metrics summarise the DSMs. The **Global Probabilistic Deviation (GPD)** measures how much the site
distributions differ overall (0 = identical, 1 = non-overlapping). The **Source Probabilistic Outlyingness
(SPO)** measures how far each site is from the latent centre of all sites.

**Requirements.** `dashi >= 0.4.2`; the optional cell below installs it. Run the notebook from the `examples` directory.

**Data.** `uci_heart_disease.csv` holds the 920 records of the four processed UCI Heart Disease files
(Janosi, Steinbrunn, Pfisterer and Detrano, UCI Machine Learning Repository, 1988) with the standard 13
attributes, the acquisition site (`domain_id`) and a binary `target` (1 = heart disease present).

In [ ]:
# Optional: install or update the requirements, then restart the kernel.
# Uncomment the next line to run it.
# %pip install -U "dashi>=0.4.2" nbformat

In [ ]:
from pathlib import Path

import dashi as ds
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42

## 1. Load and format the data

In [ ]:
DATA_PATH = Path("uci_heart_disease.csv")

raw = pd.read_csv(DATA_PATH)
raw["diagnosis"] = raw["target"].map({0: "No disease", 1: "Disease"})
print(f"{len(raw):,} records from {raw['domain_id'].nunique()} sites")
raw.head()

`format_data` casts every column to the type `dashi` expects. The coded clinical variables (sex, chest-pain
type `cp`, fasting blood sugar `fbs`, resting ECG `restecg`, exercise-induced angina `exang`, ST slope `slope`,
number of major vessels `ca` and thallium test `thal`) are declared as categorical so that no artificial distance
is imposed between their codes.

In [ ]:
SOURCE = "domain_id"
LABEL = "diagnosis"
numerical_covariates = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_covariates = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
covariates = numerical_covariates + categorical_covariates
analysis_variables = [LABEL] + covariates

data = ds.format_data(
    raw[[SOURCE] + analysis_variables],
    source_column_name=SOURCE,
    numerical_column_names=numerical_covariates,
    categorical_column_names=[SOURCE, LABEL] + categorical_covariates,
)
data.dtypes.rename("dtype").to_frame()

## 2. Cohort composition by site

The disease prevalence ranges from about a third of the patients in Hungary to more than 90% in Switzerland, and
several variables were barely recorded outside Cleveland.

In [ ]:
site_summary = pd.crosstab(raw[SOURCE], raw[LABEL])
site_summary["n"] = site_summary.sum(axis=1)
site_summary["prevalence"] = (site_summary["Disease"] / site_summary["n"]).round(2)
site_summary

In [ ]:
# Fraction of missing values per site
raw[covariates].isna().groupby(raw[SOURCE]).mean().round(2)

## 3. Unsupervised characterisation: Data Source Maps

A Data Source Map estimates the probability distribution of a variable in every site over a common support.
Numerical variables are smoothed with a kernel density estimate; categorical variables use their categories.

In [ ]:
source_maps = ds.estimate_univariate_data_source_map(
    data=data,
    source_column=SOURCE,
    numeric_smoothing=True,
)
list(source_maps)

In [ ]:
ds.plot_univariate_data_source_map(
    source_maps, variable_name=LABEL, title="Diagnosis by site",
).show()

ds.plot_univariate_data_source_map(
    source_maps, variable_name="age", title="Age by site",
).show()

## 4. Multi-Source Variability metrics

`estimate_MSV_metrics` embeds the pairwise Jensen-Shannon distances between the site distributions and returns
the GPD of the variable and the SPO of every site.

In [ ]:
def summarize_msv(source_maps, variables):
    """Return the MSV metrics, a GPD ranking and a site-by-variable SPO table."""
    metrics = {}
    gpd_rows = []
    spo_columns = []
    for variable in variables:
        result = ds.estimate_MSV_metrics(source_maps, variable_name=variable)
        metrics[variable] = result
        gpd_rows.append({"variable": variable, "GPD": float(result.GPD)})
        spo_columns.append(pd.Series(result.SPO, index=result.sources, name=variable))
    gpd = pd.DataFrame(gpd_rows).sort_values("GPD", ascending=False, ignore_index=True)
    spo = pd.concat(spo_columns, axis=1).rename_axis("site")
    return metrics, gpd, spo


msv_metrics, gpd, spo = summarize_msv(source_maps, analysis_variables)
gpd.round(3)

In [ ]:
fig = px.imshow(
    spo, aspect="auto", color_continuous_scale="Spectral_r",
    labels={"color": "SPO"}, template="plotly_white", title="Source Probabilistic Outlyingness (SPO) by site and variable",
)
fig.update_xaxes(side="top")
fig.show()

### 4.1 The metrics point to a data-quality problem

Serum cholesterol (`chol`) is the most variable covariate and Switzerland is almost a complete outlier for it
(SPO close to 1). The DSM shows why:

In [ ]:
ds.plot_univariate_data_source_map(
    source_maps, variable_name="chol", title="Serum cholesterol by site",
).show()

ds.plot_MSV(msv_metrics["chol"], dimensions=2).show()

# Records with a cholesterol value of exactly zero
(raw["chol"] == 0).groupby(raw[SOURCE]).agg(["sum", "mean"]).rename(columns={"sum": "n_zero", "mean": "fraction_zero"})

Every Swiss record, and a quarter of the Long Beach ones, has a cholesterol of exactly 0 mg/dl. That is not a
physiological value: these sites did not measure cholesterol and the gap was stored as zero. No summary
statistic of the pooled data would flag this as clearly, and a model trained on the pooled data would silently
learn "cholesterol = 0" as a proxy for the hospital.

**Reading the rest of the ranking.**

- `ca`, `thal` and `slope` also rank high, but outside Cleveland they are estimated from very few non-missing
  values (see the missingness table), so their GPD reflects incomplete recording more than patient differences.
- `fbs` and `restecg` show real differences in how the tests were recorded or performed.
- **Diagnosis** has a GPD of about 0.32: a clear **prior shift**, with Switzerland as the most distant site.
- Demographics and vital signs (`age`, `sex`, `trestbps`) are comparatively stable.

## 5. Joint (multivariate) covariate variability

Univariate maps do not show whether *combinations* of covariates differ. The multivariate DSM first embeds the
covariates with Factor Analysis of Mixed Data (FAMD, suitable for numerical plus categorical variables) and then
estimates each site's density in the reduced space.

Dimensionality reduction needs complete rows. Given the findings above, the joint analysis uses the seven
covariates that were recorded at all four sites, and the retained sample is reported.

In [ ]:
joint_covariates = ["age", "sex", "cp", "trestbps", "fbs", "restecg", "thalach"]

joint_data = data[[SOURCE] + joint_covariates].dropna().copy()
print(f"Complete cases: {len(joint_data):,} of {len(data):,} records, "
      f"{joint_data[SOURCE].nunique()} of {data[SOURCE].nunique()} sites")

joint_map = ds.estimate_multivariate_data_source_map(
    data=joint_data,
    source_column_name=SOURCE,
    dim_reduction="FAMD",
    dimensions=2,
    kde_resolution=50,
)

In [ ]:
joint_metrics = ds.estimate_MSV_metrics(joint_map)
print(f"Joint covariate GPD: {joint_metrics.GPD:.3f}")

joint_spo = pd.DataFrame({
    "site": joint_metrics.sources,
    "n": np.asarray(joint_metrics.nBySource).astype(int),
    "SPO": joint_metrics.SPO,
}).sort_values("SPO", ascending=False, ignore_index=True)
joint_spo.round(3)

In [ ]:
ds.plot_multivariate_data_source_map(joint_map).show()
ds.plot_MSV(joint_metrics, dimensions=2).show()

Even with the commonly recorded covariates only, the four hospitals remain clearly apart (joint GPD ≈ 0.59),
with no site acting as a "typical" reference for the others.

## 6. Is the variability explained by the prevalence?

Sicker cohorts naturally have different covariates, so part of the covariate shift could be a consequence of the
prior shift. Label-conditional DSMs estimate the distributions *within each diagnosis*; `estimate_MSV_metrics`
then combines the per-diagnosis distances.

In [ ]:
covariates = joint_covariates  # compare the covariates recorded at every site

conditional_maps = ds.estimate_conditional_univariate_data_source_map(
    data=data[[SOURCE, LABEL] + joint_covariates],
    source_column_name=SOURCE,
    label_column_name=LABEL,
    numeric_smoothing=True,
)

In [ ]:
_, conditional_gpd, conditional_spo = summarize_msv(conditional_maps, covariates)

comparison = (
    gpd[gpd["variable"].isin(covariates)]
    .rename(columns={"GPD": "GPD_unconditional"})
    .merge(conditional_gpd.rename(columns={"GPD": "GPD_given_diagnosis"}), on="variable")
)
comparison.round(3)

In [ ]:
ds.plot_conditional_univariate_data_source_map(
    conditional_maps, variable_name="thalach",
    title="Maximum heart rate by site, within each diagnosis",
).show()

The GPD values stay at a similar level after conditioning on the diagnosis (some slightly lower, some slightly
higher), so the differing prevalence does not explain the covariate shift: patients with the same diagnosis still look different from one hospital to another. The
joint analysis agrees:

In [ ]:
conditional_joint_data = data[[SOURCE, LABEL] + joint_covariates].dropna().copy()

conditional_joint_maps = ds.estimate_conditional_data_source_map(
    data=conditional_joint_data,
    source_column_name=SOURCE,
    label_column_name=LABEL,
    dim_reduction="FAMD",
    dimensions=2,
    kde_resolution=50,
)

In [ ]:
conditional_joint_metrics = ds.estimate_MSV_metrics(conditional_joint_maps)
print(f"Joint covariate GPD:                 {joint_metrics.GPD:.3f}")
print(f"Joint covariate GPD given diagnosis: {conditional_joint_metrics.GPD:.3f}")

for figure in ds.plot_conditional_data_source_map(conditional_joint_maps):
    figure.show()
ds.plot_MSV(conditional_joint_metrics, dimensions=2).show()

## 7. Supervised characterisation: does a model transfer between sites?

`estimate_multibatch_models` trains one model per site and tests it on a held-out 20% of **every** site. The
diagonal of the resulting matrix is the within-site performance and the off-diagonal cells measure transfer to
other sites.

Each model is a Random Forest that predicts the diagnosis. Following the unsupervised findings, the inputs are
restricted to variables recorded at all sites: `chol` (zero-coded), `ca`, `thal`, `slope` and `fbs` (mostly
missing at some site) are left out, and complete cases are used.

In [ ]:
def order_for_stratified_split(frame, source_column, label_column):
    """Place a stratified 80% of every site before its remaining 20%.

    dashi tests each model on the last 20% of the rows of every site (no shuffling),
    so this ordering turns that built-in split into a stratified hold-out.
    """
    parts = []
    for _, site_rows in frame.groupby(source_column, sort=True):
        train_index, test_index = train_test_split(
            site_rows.index.to_numpy(), test_size=0.20,
            stratify=site_rows[label_column], random_state=RANDOM_STATE,
        )
        parts.append(site_rows.loc[np.r_[train_index, test_index]])
    return pd.concat(parts, ignore_index=True)

In [ ]:
model_numerical = ["age", "trestbps", "thalach", "oldpeak"]
model_categorical = ["sex", "cp", "restecg", "exang"]

supervised_data = raw[[SOURCE, LABEL] + model_numerical + model_categorical].dropna()
supervised_data = order_for_stratified_split(supervised_data, SOURCE, LABEL)
supervised_data = ds.format_data(
    supervised_data,
    source_column_name=SOURCE,
    numerical_column_names=model_numerical,
    categorical_column_names=[LABEL] + model_categorical,
)
pd.crosstab(supervised_data[SOURCE], supervised_data[LABEL], margins=True)

In [ ]:
performance = ds.estimate_multibatch_models(
    data=supervised_data,
    inputs_numerical_column_names=model_numerical,
    inputs_categorical_column_names=model_categorical,
    output_classification_column_name=LABEL,
    source_column_name=SOURCE,
    learning_strategy="from_scratch",
    model_type="random_forest",
)
print(f"{len(performance)} training-site / test-site combinations evaluated")

In [ ]:
auc = ds.arrange_performance_metrics(metrics=performance, metric_name="AUC_MACRO")
# Rows are labelled 'From <site> to <site>' for single-site training batches.
auc.index = [label.removeprefix("From ").split(" to ")[0] for label in auc.index]
auc = auc.rename_axis(index="training site", columns="test site")
auc.round(2)

In [ ]:
ds.plot_multibatch_performance(
    metrics=performance, metric_name="AUC_MACRO",
    batching_type="source", apply_biclustering_on_source=False,
).show()

ds.plot_multibatch_performance(
    metrics=performance, metric_name="F1-SCORE_MACRO",
    batching_type="source", apply_biclustering_on_source=False,
).show()

In [ ]:
transfer = pd.DataFrame({
    "within_site_AUC": [auc.loc[site, site] for site in auc.index],
    "mean_AUC_on_other_sites": [auc.loc[site].drop(site).mean() for site in auc.index],
}, index=auc.index)
transfer["difference"] = transfer["mean_AUC_on_other_sites"] - transfer["within_site_AUC"]
transfer.round(2)

**Reading the matrix.** Rows are training sites and columns are test sites.

- **Cleveland and Hungary** yield good models on their own patients (AUC ≈ 0.9) and the Cleveland model even
  transfers to Hungary. Both, however, lose about 0.2 AUC on average when moved to the other hospitals.
- **Long Beach is hard for every model**, including its own (AUC ≈ 0.4-0.6). Whatever the training site, the
  relationship between inputs and diagnosis learned elsewhere does not hold there.
- **The Swiss model** performs poorly on its own held-out patients, but that cell rests on only two patients
  without disease, so it is not a reliable estimate. The same caveat applies to the whole Swiss column.
- The matrix is not symmetric: transferring from A to B is not the same as from B to A.

The unsupervised analysis showed *what* differs between the hospitals; the supervised one shows the
consequence: no model trained at a single hospital can be assumed to keep its performance at the others.

## 8. Take-home messages

1. **Prior shift**: disease prevalence differs markedly between hospitals.
2. **Covariate shift, partly a data-quality issue**: the largest differences come from how each hospital
   recorded the data (zero-coded cholesterol, tests that were not performed), which `dashi` exposes directly.
3. **The shift is not explained by prevalence**: it persists within each diagnosis.
4. **Performance is site-dependent**, so a model validated at a single hospital should not be assumed to hold
   elsewhere without a site-aware evaluation.

GPD and SPO are descriptive distances between estimated distributions, not significance tests. The supervised
results come from a single hold-out split of small sites and are meant to illustrate the workflow.